# 🎨 Control Estadístico de Procesos (SPC) — Cabina de Pintura Electrostática

**Planta de ensamblaje automotriz**

**Proceso monitoreado:** Pintura electrostática — control de color y acabado bajo tolerancias OEM (BMW, VW)

---

## 📐 Fundamento teórico: Control Estadístico de Procesos (SPC)

El **Control Estadístico de Procesos (SPC)** es una metodología que utiliza herramientas estadísticas
para monitorear y controlar un proceso, asegurando que opere en su máximo potencial y dentro de los
límites naturales de variación (causas comunes), y no por causas especiales o asignables.

### Límites de control a 3-sigma (3σ)

Para una característica de calidad medida (en este caso, **L\*** — luminosidad del color CIE-Lab),
se calculan tres líneas de referencia a partir de una **fase de calentamiento** (las primeras 30
mediciones del turno, cuando el proceso se asume estable):

- **Línea Central (CL):** la media (x̄) de las mediciones de calentamiento.
- **Límite de Control Superior (UCL):** `CL + 3σ`
- **Límite de Control Inferior (LCL):** `CL - 3σ`

Bajo el supuesto de una distribución aproximadamente normal, el **99.73%** de las mediciones de un
proceso bajo control caerán dentro de estos límites. Un punto fuera de ±3σ indica una **causa
especial** que requiere investigación inmediata (Regla de Western Electric #1).

### Reglas de Western Electric (detección de deriva)

Además de puntos fuera de los límites de control, existen patrones que indican que el proceso se
está desviando **antes** de salirse de control:

- **Regla usada en este notebook:** 3 puntos consecutivos del mismo lado de la línea central →
  indica un corrimiento sistemático (deriva), típicamente causado por desgaste mecánico gradual
  (p. ej. boquillas de la campana rotativa obstruyéndose).

### Gráfico de Rango Móvil (Moving Range)

Como no se toman subgrupos (una medición espectrofotométrica por intervalo), se usa una **carta
Individuals-Moving Range (I-MR)**. El Rango Móvil (`MR_i = |x_i - x_{i-1}|`) mide la variabilidad
entre mediciones consecutivas y es sensible a inestabilidad de corto plazo (p. ej. salpicaduras,
outliers por cambio de lote de pintura).

### First Time Through (FTT)

El indicador **FTT** mide el porcentaje de piezas/mediciones que cumplen especificación **a la
primera**, sin necesidad de retrabajo. Aquí se calcula sobre la última hora de producción (12
mediciones) comparando L\* contra la tolerancia de ingeniería alrededor del valor objetivo
(`Target_L`), definida por el estándar OEM del cliente.


In [1]:
# ==========================================================================
# LIBRERÍAS
# ==========================================================================
import numpy as np
import pandas as pd
import sqlite3
import json
from datetime import datetime, timedelta

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Configuración de reproducibilidad
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

pd.set_option('display.max_columns', None)
print("✅ Librerías cargadas correctamente")


✅ Librerías cargadas correctamente


## Simulación de Datos de Espectrofotómetro

Se simulan mediciones cada **5 minutos** durante un **turno de 8 horas** (96 mediciones) para
**3 cabinas de pintura**. Cada cabina pinta un color OEM distinto (proveedor y objetivo de color
propios), definidos en `BOOTHS_CONFIG`.

**Escenario de falla programado — Cabina 1:** a partir de la hora 3 del turno, `L*` disminuye
gradualmente 0.5 unidades por hora, simulando **boquillas de la campana rotativa obstruyéndose**
progresivamente (deriva mecánica real y común en pintura electrostática).

**Outliers por cambio de lote:** cada vez que una cabina cambia de lote de pintura (aprox. cada 3
horas), se inyectan **2 mediciones "salvajes"** inmediatamente después del cambio, simulando la
inestabilidad de mezcla/agitación de un lote recién instalado.


In [2]:
# ==========================================================================
# CONFIGURACIÓN DE CABINAS (colores OEM objetivo por cabina)
# ==========================================================================
BOOTHS_CONFIG = {
    1: {'proveedor': 'PPG Industries',           'target_L': 45.2, 'target_a': -0.8, 'target_b': 1.5,  'color_nombre': 'Gris Grafito OEM'},
    2: {'proveedor': 'Axalta Coating Systems',    'target_L': 38.7, 'target_a':  0.3, 'target_b': -2.1, 'color_nombre': 'Azul Nocturno OEM'},
    3: {'proveedor': 'BASF Coatings',             'target_L': 52.4, 'target_a': -1.2, 'target_b': 3.8,  'color_nombre': 'Plata Ártico OEM'},
}

SHIFT_START = datetime(2026, 8, 18, 6, 0, 0)   # inicio de turno
SHIFT_DURATION_MIN = 8 * 60                     # 8 horas en minutos
SAMPLE_INTERVAL_MIN = 5                         # una medición cada 5 minutos
SPEC_TOLERANCE_L = 1.0                          # tolerancia de ingeniería OEM alrededor de Target_L

# Tiempos (en minutos desde el inicio del turno) en los que cada cabina cambia de lote de pintura
BATCH_CHANGE_TIMES_MIN = [0, 180, 360]          # cada 3 horas


def simulate_paint_data(booths_config=BOOTHS_CONFIG, seed=RANDOM_SEED):
    """
    Genera mediciones simuladas de espectrofotómetro (L*, a*, b*, Gloss_Units)
    cada 5 minutos durante un turno de 8 horas para cada cabina de pintura.

    Incluye:
      - Deriva programada en Cabina 1 (boquillas obstruidas) a partir de la hora 3.
      - 2 mediciones outlier por cada cambio de lote de pintura.

    Retorna un DataFrame con trazabilidad completa por Booth_ID y Timestamp.
    """
    local_rng = np.random.default_rng(seed)
    tiempos_min = np.arange(0, SHIFT_DURATION_MIN, SAMPLE_INTERVAL_MIN)
    n = len(tiempos_min)

    registros = []
    batch_counter_global = 1000  # para generar números de lote únicos

    for booth_id, cfg in booths_config.items():
        # --- Ruido normal del proceso (causas comunes) ---
        L = cfg['target_L'] + local_rng.normal(0, 0.15, n)
        a = cfg['target_a'] + local_rng.normal(0, 0.10, n)
        b = cfg['target_b'] + local_rng.normal(0, 0.12, n)
        gloss = 85.0 + local_rng.normal(0, 1.0, n)   # brillo base ~85 GU (Gloss Units)

        # --- Deriva programada: Cabina 1, boquillas obstruidas desde la hora 3 ---
        if booth_id == 1:
            horas_transcurridas = tiempos_min / 60.0
            deriva = -0.5 * np.clip(horas_transcurridas - 3.0, 0, None)  # -0.5 L*/hora desde hora 3
            L = L + deriva

        # --- Asignación de lote de pintura activo para cada medición ---
        batch_ids = np.zeros(n, dtype=int)
        for i, t in enumerate(tiempos_min):
            lote_idx = sum(t >= ct for ct in BATCH_CHANGE_TIMES_MIN) - 1
            batch_ids[i] = lote_idx

        # --- Outliers "salvajes": 2 por cada cambio de lote (excepto el lote inicial) ---
        for ct in BATCH_CHANGE_TIMES_MIN[1:]:
            idx_cambio = int(np.searchsorted(tiempos_min, ct))
            idxs_outlier = [i for i in (idx_cambio, idx_cambio + 1) if i < n]
            for oi in idxs_outlier:
                signo = local_rng.choice([-1, 1])
                L[oi] += signo * local_rng.uniform(3.0, 5.0)
                a[oi] += local_rng.choice([-1, 1]) * local_rng.uniform(1.0, 2.0)
                b[oi] += local_rng.choice([-1, 1]) * local_rng.uniform(1.0, 2.0)
                gloss[oi] += local_rng.choice([-1, 1]) * local_rng.uniform(5.0, 10.0)

        timestamps = [SHIFT_START + timedelta(minutes=int(t)) for t in tiempos_min]

        for i in range(n):
            registros.append({
                'Timestamp': timestamps[i],
                'Booth_ID': booth_id,
                'L': round(float(L[i]), 3),
                'a': round(float(a[i]), 3),
                'b': round(float(b[i]), 3),
                'Gloss_Units': round(float(gloss[i]), 2),
                'Batch_Index': int(batch_ids[i]),
            })

    df = pd.DataFrame(registros).sort_values(['Booth_ID', 'Timestamp']).reset_index(drop=True)
    return df


df_mediciones = simulate_paint_data()
print(f"✅ {len(df_mediciones)} mediciones simuladas para {df_mediciones['Booth_ID'].nunique()} cabinas")
df_mediciones.head(10)


✅ 288 mediciones simuladas para 3 cabinas


,Timestamp,Booth_ID,L,a,b,Gloss_Units,Batch_Index
0,2026-08-18 06:00:00,1,45.246,-0.932,1.454,84.71,0
1,2026-08-18 06:05:00,1,45.044,-0.900,1.675,84.90,0
2,2026-08-18 06:10:00,1,45.313,-0.760,1.367,84.75,0
3,2026-08-18 06:15:00,1,45.341,-0.891,1.393,85.15,0
4,2026-08-18 06:20:00,1,44.907,-0.838,1.577,86.47,0
5,2026-08-18 06:25:00,1,45.005,-0.670,1.453,82.43,0
6,2026-08-18 06:30:00,1,45.219,-0.836,1.499,84.76,0
7,2026-08-18 06:35:00,1,45.153,-0.726,1.480,85.18,0
8,2026-08-18 06:40:00,1,45.197,-0.893,1.541,85.30,0
9,2026-08-18 06:45:00,1,45.072,-0.821,1.669,84.63,0


## Base de Datos SQLite de Trazabilidad

Se crea una base de datos `paint_qc.db` con dos tablas:

- **`paint_batch_metadata`**: relaciona cada `Booth_ID` con su `Paint_Supplier`, `Batch_Number` y
  los valores objetivo de fábrica (`Target_L`, `Target_a`, `Target_b`) — necesarios para trazar
  cualquier no-conformidad hasta el lote de pintura exacto.
- **`paint_measurements`**: histórico completo de mediciones del espectrofotómetro (persistencia
  de los datos simulados, tal como llegarían de un instrumento real vía OPC-UA/MQTT).


In [3]:
# ==========================================================================
# BASE DE DATOS SQLITE — TRAZABILIDAD DE LOTES Y MEDICIONES
# ==========================================================================
DB_PATH = 'paint_qc.db'
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('DROP TABLE IF EXISTS paint_batch_metadata')
cursor.execute('''
    CREATE TABLE paint_batch_metadata (
        Booth_ID        INTEGER NOT NULL,
        Batch_Index     INTEGER NOT NULL,
        Paint_Supplier  TEXT NOT NULL,
        Batch_Number    TEXT NOT NULL,
        Color_Nombre    TEXT NOT NULL,
        Target_L        REAL NOT NULL,
        Target_a        REAL NOT NULL,
        Target_b        REAL NOT NULL,
        Batch_Start_Time TEXT NOT NULL,
        PRIMARY KEY (Booth_ID, Batch_Index)
    )
''')

# --- Generar metadata de lotes: cada cabina usa 3 lotes distintos durante el turno ---
registros_metadata = []
for booth_id, cfg in BOOTHS_CONFIG.items():
    for lote_idx, inicio_min in enumerate(BATCH_CHANGE_TIMES_MIN):
        numero_lote = f"{cfg['proveedor'].split()[0].upper()}-B{booth_id}-LOT{2400 + booth_id*10 + lote_idx}"
        inicio_ts = (SHIFT_START + timedelta(minutes=inicio_min)).isoformat()
        registros_metadata.append((
            booth_id, lote_idx, cfg['proveedor'], numero_lote, cfg['color_nombre'],
            cfg['target_L'], cfg['target_a'], cfg['target_b'], inicio_ts
        ))

cursor.executemany('''
    INSERT INTO paint_batch_metadata
    (Booth_ID, Batch_Index, Paint_Supplier, Batch_Number, Color_Nombre, Target_L, Target_a, Target_b, Batch_Start_Time)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
''', registros_metadata)

# --- Persistir mediciones ---
df_mediciones_sql = df_mediciones.copy()
df_mediciones_sql['Timestamp'] = df_mediciones_sql['Timestamp'].astype(str)
df_mediciones_sql.to_sql('paint_measurements', conn, if_exists='replace', index=False)

conn.commit()

df_batch_metadata = pd.read_sql('SELECT * FROM paint_batch_metadata ORDER BY Booth_ID, Batch_Index', conn)
print(f"✅ Base de datos '{DB_PATH}' creada — {len(df_batch_metadata)} lotes registrados, {len(df_mediciones_sql)} mediciones persistidas")
df_batch_metadata


✅ Base de datos 'paint_qc.db' creada — 9 lotes registrados, 288 mediciones persistidas


,Booth_ID,Batch_Index,Paint_Supplier,Batch_Number,Color_Nombre,Target_L,Target_a,Target_b,Batch_Start_Time
0,1,0,PPG Industries,PPG-B1-LOT2410,Gris Grafito OEM,45.2,-0.8,1.5,2026-08-18T06:00:00
1,1,1,PPG Industries,PPG-B1-LOT2411,Gris Grafito OEM,45.2,-0.8,1.5,2026-08-18T09:00:00
2,1,2,PPG Industries,PPG-B1-LOT2412,Gris Grafito OEM,45.2,-0.8,1.5,2026-08-18T12:00:00
3,2,0,Axalta Coating Systems,AXALTA-B2-LOT2420,Azul Nocturno OEM,38.7,0.3,-2.1,2026-08-18T06:00:00
4,2,1,Axalta Coating Systems,AXALTA-B2-LOT2421,Azul Nocturno OEM,38.7,0.3,-2.1,2026-08-18T09:00:00
5,2,2,Axalta Coating Systems,AXALTA-B2-LOT2422,Azul Nocturno OEM,38.7,0.3,-2.1,2026-08-18T12:00:00
6,3,0,BASF Coatings,BASF-B3-LOT2430,Plata Ártico OEM,52.4,-1.2,3.8,2026-08-18T06:00:00
7,3,1,BASF Coatings,BASF-B3-LOT2431,Plata Ártico OEM,52.4,-1.2,3.8,2026-08-18T09:00:00
8,3,2,BASF Coatings,BASF-B3-LOT2432,Plata Ártico OEM,52.4,-1.2,3.8,2026-08-18T12:00:00


## Cálculo de Límites de Control (Carta Individuals — I-MR)

Para cada cabina, la **fase de calentamiento** son las primeras **30 mediciones** del turno
(2.5 horas), asumidas representativas del proceso bajo control. A partir de ahí se fija:

`UCL = x̄ + 3σ` &nbsp;&nbsp; `CL = x̄` &nbsp;&nbsp; `LCL = x̄ - 3σ`

Estos límites permanecen **fijos** durante el resto del turno para poder detectar la deriva real.


In [4]:
# ==========================================================================
# LÍMITES DE CONTROL (3-SIGMA) A PARTIR DE LA FASE DE CALENTAMIENTO
# ==========================================================================
N_CALENTAMIENTO = 30

def calcular_limites_control(df_booth, n_calentamiento=N_CALENTAMIENTO):
    """Calcula CL, UCL y LCL de L* usando las primeras n mediciones (fase de calentamiento)."""
    fase_calentamiento = df_booth.sort_values('Timestamp').head(n_calentamiento)['L']
    cl = fase_calentamiento.mean()
    sigma = fase_calentamiento.std(ddof=1)
    return {
        'CL': cl,
        'UCL': cl + 3 * sigma,
        'LCL': cl - 3 * sigma,
        'sigma': sigma,
    }


def calcular_rango_movil(df_booth):
    """Calcula el Rango Móvil (MR) entre mediciones consecutivas de L*, ordenadas por tiempo."""
    serie = df_booth.sort_values('Timestamp')['L'].reset_index(drop=True)
    mr = serie.diff().abs()
    return mr


# Pre-calcular límites de control para las 3 cabinas
limites_por_cabina = {
    booth_id: calcular_limites_control(df_mediciones[df_mediciones['Booth_ID'] == booth_id])
    for booth_id in BOOTHS_CONFIG
}

for booth_id, lim in limites_por_cabina.items():
    print(f"Cabina {booth_id} -> CL={lim['CL']:.3f}  UCL={lim['UCL']:.3f}  LCL={lim['LCL']:.3f}  (σ={lim['sigma']:.3f})")


Cabina 1 -> CL=45.203  UCL=45.552  LCL=44.853  (σ=0.117)
Cabina 2 -> CL=38.663  UCL=39.149  LCL=38.177  (σ=0.162)
Cabina 3 -> CL=52.390  UCL=52.918  LCL=51.862  (σ=0.176)


## Lógica de Asesoramiento al Encargado de Pintura

Reglas automáticas implementadas:

1. **Regla de Western Electric (3 puntos consecutivos):** si las 3 mediciones más recientes están
   todas por encima o todas por debajo de la Línea Central → alerta de deriva sistemática.
2. **FTT < 90%:** si el First Time Through de la última hora cae por debajo del 90% → se sugiere
   revisar/cambiar los filtros de aire de la cabina.


In [5]:
# ==========================================================================
# REGLAS DE ASESORAMIENTO AUTOMÁTICO (WESTERN ELECTRIC + FTT)
# ==========================================================================

def detectar_regla_western_electric(df_booth, cl, n_puntos=3):
    """
    Regla de Western Electric: True si las últimas `n_puntos` mediciones de L*
    están todas del mismo lado (encima o debajo) de la Línea Central (CL).
    """
    serie = df_booth.sort_values('Timestamp')['L'].tail(n_puntos)
    if len(serie) < n_puntos:
        return False
    todas_encima = (serie > cl).all()
    todas_debajo = (serie < cl).all()
    return bool(todas_encima or todas_debajo)


def calcular_ftt(df_booth, target_l, tolerancia=SPEC_TOLERANCE_L, ventana_min=60):
    """
    First Time Through: % de mediciones de la última hora dentro de la tolerancia
    de especificación OEM alrededor de Target_L.
    """
    df_ordenado = df_booth.sort_values('Timestamp')
    ultimo_ts = df_ordenado['Timestamp'].max()
    ventana = df_ordenado[df_ordenado['Timestamp'] > ultimo_ts - timedelta(minutes=ventana_min)]
    if len(ventana) == 0:
        return 100.0
    dentro_spec = ventana['L'].between(target_l - tolerancia, target_l + tolerancia)
    return round(100.0 * dentro_spec.mean(), 1)


def generar_alertas(booth_id, df_booth, limites, target_l):
    """Genera la lista de alertas activas para una cabina, según las reglas de negocio."""
    alertas = []

    # Regla 1: Western Electric — 3 puntos consecutivos del mismo lado de CL
    if detectar_regla_western_electric(df_booth, limites['CL']):
        alertas.append(
            f"⚠️ SPC ALERT - Booth {booth_id} shows systematic drift in L value. "
            f"Potential clogging in rotary bell. Inspect nozzles immediately."
        )

    # Regla 2: FTT por debajo del 90%
    ftt = calcular_ftt(df_booth, target_l)
    if ftt < 90.0:
        alertas.append(
            f"🔧 MAINTENANCE REPORT - Booth {booth_id} FTT at {ftt}% (below 90% threshold). "
            f"Recommend air filter replacement in booth ventilation system."
        )

    return alertas, ftt


print("✅ Funciones de asesoramiento (Western Electric + FTT) definidas")


✅ Funciones de asesoramiento (Western Electric + FTT) definidas


## Dashboard Interactivo de Control Estadístico

El panel incluye:

- **Selector de Cabina** (`Dropdown`)
- **Gráfica de Control (Shewhart)** de L\* con CL / UCL / LCL
- **Gráfico de Rango Móvil** para variabilidad de corto plazo
- **Termómetro FTT** (gauge) — % dentro de especificación en la última hora
- **Panel de alertas** con las reglas de asesoramiento automático


In [7]:
# ==========================================================================
# DASHBOARD INTERACTIVO — SELECTOR DE CABINA + SPC + MR + FTT + ALERTAS
# ==========================================================================

selector_cabina = widgets.Dropdown(
    options=[(f"Cabina {bid} — {cfg['color_nombre']} ({cfg['proveedor']})", bid)
             for bid, cfg in BOOTHS_CONFIG.items()],
    value=1,
    description='Cabina:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='480px')
)

salida_dashboard = widgets.Output()


def construir_grafico_control(df_booth, limites, booth_id):
    """Construye la carta de control Shewhart de L* con bandas UCL/CL/LCL."""
    df_ordenado = df_booth.sort_values('Timestamp')
    fuera_control = (df_ordenado['L'] > limites['UCL']) | (df_ordenado['L'] < limites['LCL'])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_ordenado['Timestamp'], y=df_ordenado['L'],
        mode='lines+markers', name='L* medido',
        line=dict(color='#2563eb', width=1.5),
        marker=dict(
            size=6,
            color=np.where(fuera_control, '#dc2626', '#2563eb'),
        )
    ))
    for nombre, valor, color, dash in [
        ('UCL', limites['UCL'], '#dc2626', 'dash'),
        ('CL',  limites['CL'],  '#16a34a', 'solid'),
        ('LCL', limites['LCL'], '#dc2626', 'dash'),
    ]:
        fig.add_hline(y=valor, line_color=color, line_dash=dash, line_width=1.5,
                       annotation_text=f"{nombre}={valor:.2f}", annotation_position='right')

    fig.update_layout(
        title=f'Carta de Control Shewhart — L* (Cabina {booth_id})',
        xaxis_title='Tiempo', yaxis_title='L* (Luminosidad)',
        height=380, margin=dict(t=50, b=40, l=50, r=110),
        template='plotly_white', showlegend=False
    )
    return fig


def construir_grafico_rango_movil(df_booth, booth_id):
    """Construye el gráfico de Rango Móvil (variabilidad entre mediciones consecutivas)."""
    df_ordenado = df_booth.sort_values('Timestamp').reset_index(drop=True)
    mr = calcular_rango_movil(df_booth)
    mr_bar = mr.mean()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_ordenado['Timestamp'], y=mr,
        mode='lines+markers', name='Rango Móvil',
        line=dict(color='#f59e0b', width=1.5), marker=dict(size=5)
    ))
    fig.add_hline(y=mr_bar, line_color='#16a34a', line_dash='solid',
                   annotation_text=f'MR̄={mr_bar:.3f}', annotation_position='right')
    fig.add_hline(y=mr_bar * 3.267, line_color='#dc2626', line_dash='dash',
                   annotation_text='UCL (MR)', annotation_position='right')

    fig.update_layout(
        title=f'Rango Móvil — Variabilidad entre Mediciones (Cabina {booth_id})',
        xaxis_title='Tiempo', yaxis_title='|ΔL*|',
        height=300, margin=dict(t=50, b=40, l=50, r=110),
        template='plotly_white', showlegend=False
    )
    return fig


def construir_gauge_ftt(ftt_pct, booth_id):
    """Construye el termómetro (gauge) de First Time Through."""
    color_barra = '#16a34a' if ftt_pct >= 90 else ('#f59e0b' if ftt_pct >= 75 else '#dc2626')
    fig = go.Figure(go.Indicator(
        mode='gauge+number',
        value=ftt_pct,
        number={'suffix': '%'},
        title={'text': f'FTT — Cabina {booth_id}<br><span style="font-size:0.7em">Última hora</span>'},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': color_barra},
            'steps': [
                {'range': [0, 75], 'color': '#fee2e2'},
                {'range': [75, 90], 'color': '#fef3c7'},
                {'range': [90, 100], 'color': '#dcfce7'},
            ],
            'threshold': {'line': {'color': '#111827', 'width': 3}, 'value': 90}
        }
    ))
    fig.update_layout(height=300, margin=dict(t=60, b=10, l=30, r=30))
    return fig


def refrescar_dashboard(booth_id):
    """Redibuja todos los componentes del dashboard para la cabina seleccionada."""
    with salida_dashboard:
        clear_output(wait=True)

        cfg = BOOTHS_CONFIG[booth_id]
        df_booth = df_mediciones[df_mediciones['Booth_ID'] == booth_id]
        limites = limites_por_cabina[booth_id]

        alertas, ftt_pct = generar_alertas(booth_id, df_booth, limites, cfg['target_L'])

        display(HTML(f"""
        <div style='padding:8px 14px;background:#f1f5f9;border-radius:8px;margin-bottom:8px;font-family:sans-serif'>
          <b>Cabina {booth_id}</b> — {cfg['color_nombre']} · Proveedor: {cfg['proveedor']} ·
          Target L*={cfg['target_L']} a*={cfg['target_a']} b*={cfg['target_b']}
        </div>
        """))

        fig_control = construir_grafico_control(df_booth, limites, booth_id)
        fig_mr = construir_grafico_rango_movil(df_booth, booth_id)
        fig_gauge = construir_gauge_ftt(ftt_pct, booth_id)

        display(widgets.HBox([
            widgets.VBox([go.FigureWidget(fig_control), go.FigureWidget(fig_mr)]),
            go.FigureWidget(fig_gauge)
        ]))

        if alertas:
            html_alertas = "<div style='font-family:sans-serif'>"
            for alerta in alertas:
                html_alertas += (
                    f"<div style='padding:10px 14px;background:#fef2f2;border-left:4px solid #dc2626;"
                    f"border-radius:4px;margin-top:6px;color:#7f1d1d'>{alerta}</div>"
                )
            html_alertas += "</div>"
            display(HTML(html_alertas))
        else:
            display(HTML(
                "<div style='padding:10px 14px;background:#f0fdf4;border-left:4px solid #16a34a;"
                "border-radius:4px;margin-top:6px;color:#14532d;font-family:sans-serif'>"
                "✅ Sin alertas activas — proceso dentro de control estadístico.</div>"
            ))


def _on_cambio_cabina(change):
    if change['name'] == 'value':
        refrescar_dashboard(change['new'])


selector_cabina.observe(_on_cambio_cabina, names='value')

display(widgets.VBox([selector_cabina, salida_dashboard]))
refrescar_dashboard(selector_cabina.value)


## Resumen Ejecutivo del Turno

Se evalúan las 3 cabinas al cierre del turno y se identifica la **cabina más crítica** (mayor
desviación absoluta de L\* respecto a su límite de control), junto con la magnitud de la
desviación máxima detectada. El resumen se persiste en `paint_qc_summary.json` para su consumo
por sistemas MES/dashboards corporativos.


In [8]:
# ==========================================================================
# RESUMEN FINAL — CABINA MÁS CRÍTICA Y DESVIACIÓN MÁXIMA (JSON)
# ==========================================================================

resumen_cabinas = {}
for booth_id, cfg in BOOTHS_CONFIG.items():
    df_booth = df_mediciones[df_mediciones['Booth_ID'] == booth_id]
    limites = limites_por_cabina[booth_id]
    alertas, ftt_pct = generar_alertas(booth_id, df_booth, limites, cfg['target_L'])

    desviacion_max_ucl = float((df_booth['L'] - limites['UCL']).max())
    desviacion_max_lcl = float((limites['LCL'] - df_booth['L']).max())
    desviacion_max = max(desviacion_max_ucl, desviacion_max_lcl, 0.0)

    resumen_cabinas[booth_id] = {
        'Booth_ID': booth_id,
        'Color_Nombre': cfg['color_nombre'],
        'Paint_Supplier': cfg['proveedor'],
        'CL': round(limites['CL'], 3),
        'UCL': round(limites['UCL'], 3),
        'LCL': round(limites['LCL'], 3),
        'Desviacion_Maxima_L': round(desviacion_max, 3),
        'FTT_Ultima_Hora_pct': ftt_pct,
        'Alertas_Activas': alertas,
        'Total_Mediciones': int(len(df_booth)),
    }

booth_critico = max(resumen_cabinas.values(), key=lambda r: r['Desviacion_Maxima_L'])

resumen_final = {
    'Turno_Inicio': SHIFT_START.isoformat(),
    'Turno_Duracion_Horas': SHIFT_DURATION_MIN / 60,
    'Booth_Mas_Critico': booth_critico['Booth_ID'],
    'Desviacion_Maxima_Detectada_L': booth_critico['Desviacion_Maxima_L'],
    'Detalle_Por_Cabina': resumen_cabinas,
    'Generado_En': datetime.now().isoformat(),
}

with open('paint_qc_summary.json', 'w', encoding='utf-8') as f:
    json.dump(resumen_final, f, indent=2, ensure_ascii=False)

print(f"🏭 Cabina más crítica del turno: Booth {booth_critico['Booth_ID']} "
      f"({booth_critico['Color_Nombre']}) — Desviación máxima: {booth_critico['Desviacion_Maxima_L']:.3f} unidades L*")
print("✅ Resumen guardado en 'paint_qc_summary.json'")

resumen_final


🏭 Cabina más crítica del turno: Booth 1 (Gris Grafito OEM) — Desviación máxima: 5.674 unidades L*
✅ Resumen guardado en 'paint_qc_summary.json'


{'Turno_Inicio': '2026-08-18T06:00:00',
 'Turno_Duracion_Horas': 8.0,
 'Booth_Mas_Critico': 1,
 'Desviacion_Maxima_Detectada_L': 5.674,
 'Detalle_Por_Cabina': {1: {'Booth_ID': 1,
   'Color_Nombre': 'Gris Grafito OEM',
   'Paint_Supplier': 'PPG Industries',
   'CL': np.float64(45.203),
   'UCL': np.float64(45.552),
   'LCL': np.float64(44.853),
   'Desviacion_Maxima_L': 5.674,
   'FTT_Ultima_Hora_pct': np.float64(0.0),
   'Alertas_Activas': ['⚠️ SPC ALERT - Booth 1 shows systematic drift in L value. Potential clogging in rotary bell. Inspect nozzles immediately.',
    '🔧 MAINTENANCE REPORT - Booth 1 FTT at 0.0% (below 90% threshold). Recommend air filter replacement in booth ventilation system.'],
   'Total_Mediciones': 96},
  2: {'Booth_ID': 2,
   'Color_Nombre': 'Azul Nocturno OEM',
   'Paint_Supplier': 'Axalta Coating Systems',
   'CL': np.float64(38.663),
   'UCL': np.float64(39.149),
   'LCL': np.float64(38.177),
   'Desviacion_Maxima_L': 3.907,
   'FTT_Ultima_Hora_pct': np.float64

---

## 📌 Conclusiones

Este dashboard demuestra un flujo completo de **SPC industrial aplicado a pintura automotriz**:
ingesta simulada de espectrofotómetro → trazabilidad de lotes en SQLite → cálculo de límites de
control 3σ desde fase de calentamiento → detección de deriva (Western Electric) → indicador FTT →
generación de alertas accionables para el encargado de pintura → resumen ejecutivo en JSON.

La **Cabina 1** ilustra el caso de uso principal: una deriva mecánica gradual (boquillas
obstruidas) que el SPC detecta *antes* de que se acumulen piezas fuera de especificación en masa,
que es precisamente el objetivo de un sistema de monitoreo de calidad en tiempo real.
